# Agent 02 Quotes Realtime Monitor

Notebook refactorizado.

Este notebook ya no ejecuta Agent02.

Su funcion es:
- fijar el contexto del run,
- imprimir comandos de terminal para Agent02 y monitores,
- y visualizar artefactos de validacion de quotes strict.

Nota:
- el nombre historico menciona `trades`, pero la operativa realtime estabilizada aqui es la de `quotes`.

In [1]:
from pathlib import Path
import json
import pandas as pd

# =========================
# TEST RUN
# =========================
TEST_RUN_ID = "20260312_quotes_lifecycle3_test"
TEST_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / TEST_RUN_ID
TEST_QUOTES_ROOT = Path(r"D:\quotes\__pruebas__\lifecycle_prod")

# =========================
# PRODUCTION TEMPLATE
# =========================
PROD_RUN_ID = "YYYYMMDD_quotes_session_prod_01"
PROD_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / PROD_RUN_ID
PROD_QUOTES_ROOT = Path(r"D:\quotes")

AGENT_RUNNERS_DIR = Path(globals().get("AGENT_RUNNERS_DIR", r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents"))
AGENT02_PS1 = Path(globals().get("AGENT02_PS1", AGENT_RUNNERS_DIR / "run_agent02_quotes_strict_loop.ps1"))
WATCH_SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\033_watch_quotes_agent_status.py")

MAX_FILES = 50000
SLEEP_SEC = 15

print("TEST_RUN_ID:", TEST_RUN_ID)
print("TEST_RUN_DIR:", TEST_RUN_DIR)
print("TEST_QUOTES_ROOT:", TEST_QUOTES_ROOT)
print("PROD_RUN_ID:", PROD_RUN_ID)
print("PROD_RUN_DIR:", PROD_RUN_DIR)
print("PROD_QUOTES_ROOT:", PROD_QUOTES_ROOT)
print("AGENT02_PS1 exists:", AGENT02_PS1.exists())
print("WATCH_SCRIPT exists:", WATCH_SCRIPT.exists())

TEST_RUN_ID: 20260312_quotes_lifecycle3_test
TEST_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test
TEST_QUOTES_ROOT: D:\quotes\__pruebas__\lifecycle_prod
PROD_RUN_ID: YYYYMMDD_quotes_session_prod_01
PROD_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01
PROD_QUOTES_ROOT: D:\quotes
AGENT02_PS1 exists: False
WATCH_SCRIPT exists: True


## Comandos Reales de Prueba

Corrida de prueba activa:
- `RUN_ID = 20260312_quotes_lifecycle3_test`
- `QuotesRoot = D:\quotes\__pruebas__\lifecycle_prod`
- `RUN_DIR = C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test`

Artefactos principales de Agent02:
- `quotes_agent_strict_events_current.csv`
- `retry_queue_quotes_strict_current.csv`
- `live_status_quotes_strict.json`
- `run_config_quotes_strict.json`
- `batch_manifest_quotes_strict.csv`

In [2]:
test_agent02_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -QuotesRoot "{TEST_QUOTES_ROOT}" ^
  -MaxFiles {MAX_FILES} ^
  -SleepSec {SLEEP_SEC}'''

test_agent02_reset_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -QuotesRoot "{TEST_QUOTES_ROOT}" ^
  -MaxFiles {MAX_FILES} ^
  -SleepSec {SLEEP_SEC} ^
  -ResetState'''

cmd_once = (
    f'python "{WATCH_SCRIPT}" '
    f'--events "{TEST_RUN_DIR / "quotes_agent_strict_events_current.csv"}" '
    f'--state "{TEST_RUN_DIR / "quotes_agent_strict_state.json"}" --once'
)

cmd_live = (
    f'python "{WATCH_SCRIPT}" '
    f'--events "{TEST_RUN_DIR / "quotes_agent_strict_events_current.csv"}" '
    f'--state "{TEST_RUN_DIR / "quotes_agent_strict_state.json"}" --interval 10 --last-n 15'
)

print("=== TEST: AGENT02 ===")
print(test_agent02_cmd)
print()
print("=== TEST: AGENT02 RESET ===")
print(test_agent02_reset_cmd)
print()
print("=== TEST: WATCH ONCE ===")
print(cmd_once)
print()
print("=== TEST: WATCH LIVE ===")
print(cmd_live)

=== TEST: AGENT02 ===
powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -QuotesRoot "D:\quotes\__pruebas__\lifecycle_prod" ^
  -MaxFiles 50000 ^
  -SleepSec 15

=== TEST: AGENT02 RESET ===
powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -QuotesRoot "D:\quotes\__pruebas__\lifecycle_prod" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState

=== TEST: WATCH ONCE ===
python "C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\033_watch_quotes_agent_status.py" --events "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_current.csv" --state "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_ag

## Comandos Preparados de Produccion

Templates. Ajusta `PROD_RUN_ID` antes de lanzar.

In [3]:
prod_agent02_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -QuotesRoot "{PROD_QUOTES_ROOT}" ^
  -MaxFiles {MAX_FILES} ^
  -SleepSec {SLEEP_SEC}'''

prod_watch_once = (
    f'python "{WATCH_SCRIPT}" '
    f'--events "{PROD_RUN_DIR / "quotes_agent_strict_events_current.csv"}" '
    f'--state "{PROD_RUN_DIR / "quotes_agent_strict_state.json"}" --once'
)

prod_watch_live = (
    f'python "{WATCH_SCRIPT}" '
    f'--events "{PROD_RUN_DIR / "quotes_agent_strict_events_current.csv"}" '
    f'--state "{PROD_RUN_DIR / "quotes_agent_strict_state.json"}" --interval 10 --last-n 15'
)

print("=== PROD: AGENT02 ===")
print(prod_agent02_cmd)
print()
print("=== PROD: WATCH ONCE ===")
print(prod_watch_once)
print()
print("=== PROD: WATCH LIVE ===")
print(prod_watch_live)

=== PROD: AGENT02 ===
powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -QuotesRoot "D:\quotes" ^
  -MaxFiles 50000 ^
  -SleepSec 15

=== PROD: WATCH ONCE ===
python "C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\033_watch_quotes_agent_status.py" --events "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01\quotes_agent_strict_events_current.csv" --state "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01\quotes_agent_strict_state.json" --once

=== PROD: WATCH LIVE ===
python "C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\033_watch_quotes_agent_status.py" --events "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01\quotes_agent_strict_events_current.csv" --state "C:\TSIS_

## Visualizacion de Artefactos del Run

Esta parte sustituye a la antigua ejecucion de `032_probe_quotes_agent_behavior_strict.py` dentro del notebook.

In [4]:
RUN_DIR = TEST_RUN_DIR
EVENTS_HISTORY_CSV = RUN_DIR / "quotes_agent_strict_events_history.csv"
EVENTS_CURRENT_CSV = RUN_DIR / "quotes_agent_strict_events_current.csv"
RETRY_CURRENT_CSV = RUN_DIR / "retry_queue_quotes_strict_current.csv"
RETRY_FROZEN_CSV = RUN_DIR / "retry_frozen_quotes_strict.csv"
STATE_FILE = RUN_DIR / "quotes_agent_strict_state.json"
LIVE_STATUS_JSON = RUN_DIR / "live_status_quotes_strict.json"
RUN_CONFIG_JSON = RUN_DIR / "run_config_quotes_strict.json"
BATCH_MANIFEST_CSV = RUN_DIR / "batch_manifest_quotes_strict.csv"

for p in [
    EVENTS_HISTORY_CSV, EVENTS_CURRENT_CSV, RETRY_CURRENT_CSV, RETRY_FROZEN_CSV,
    STATE_FILE, LIVE_STATUS_JSON, RUN_CONFIG_JSON, BATCH_MANIFEST_CSV,
]:
    print(p, "exists", p.exists())

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_history.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\retry_queue_quotes_strict_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\retry_frozen_quotes_strict.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_state.json exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\live_status_quotes_strict.json exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\run_config_quotes_strict.json exists True
C:\TSIS_Data

In [5]:
if LIVE_STATUS_JSON.exists():
    print("\nLIVE STATUS:")
    print(json.dumps(json.loads(LIVE_STATUS_JSON.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if RUN_CONFIG_JSON.exists():
    print("\nRUN CONFIG:")
    print(json.dumps(json.loads(RUN_CONFIG_JSON.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if EVENTS_CURRENT_CSV.exists():
    ev = pd.read_csv(EVENTS_CURRENT_CSV)
    print("\nEVENTS CURRENT rows:", len(ev))
    if "severity" in ev.columns:
        print(ev["severity"].value_counts(dropna=False))
    display(ev.head(20))

if RETRY_CURRENT_CSV.exists():
    rq = pd.read_csv(RETRY_CURRENT_CSV)
    print("\nRETRY CURRENT rows:", len(rq))
    display(rq.head(20))

if BATCH_MANIFEST_CSV.exists():
    bm = pd.read_csv(BATCH_MANIFEST_CSV)
    print("\nBATCH MANIFEST rows:", len(bm))
    display(bm.head(20))


LIVE STATUS:
{
  "run_id": "20260312_quotes_lifecycle3_test",
  "updated_utc": "2026-03-12T14:33:44.999069+00:00",
  "probe_root": "D:\\quotes\\__pruebas__\\lifecycle_prod",
  "max_files": 50000,
  "files_discovered_total": 1695,
  "files_pending": 401,
  "files_processed_total_state": 176,
  "files_current_snapshot": 401,
  "severity_counts_current": {
    "HARD_FAIL": 3,
    "PASS": 176,
    "SOFT_FAIL": 222
  },
  "retry_pending_files_current": 0,
  "top_causes_current": [
    {
      "cause": "crossed_rows_present_but_under_threshold",
      "count": 222
    },
    {
      "cause": "crossed_ratio_gt_threshold",
      "count": 3
    },
    {
      "cause": "crossed_ratio_gt_hard_cap",
      "count": 1
    }
  ],
  "events_current_csv": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260312_quotes_lifecycle3_test\\quotes_agent_strict_events_current.csv",
  "retry_current_csv": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260312_quot

,file,rows,severity,issues,warns,action,crossed_rows,crossed_ratio_pct,negative_price_rows,missing_required_cols,dtype_mismatches,ask_integer_pct,bid_integer_pct,ask_eq_round_bid_pct,processed_at_utc,run_id
0,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1493,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,7,0.468855,0,[],[],6.831882,4.621567,6.630944,2026-03-12 14:33:42.157754+00:00,20260312_quotes_lifecycle3_test
1,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,225,PASS,[],[],count_coverage,0,0.000000,0,[],[],2.666667,0.444444,2.666667,2026-03-12 14:33:42.159546+00:00,20260312_quotes_lifecycle3_test
2,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1259,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.158856,0,[],[],5.003971,1.906275,5.003971,2026-03-12 14:33:42.161375+00:00,20260312_quotes_lifecycle3_test
3,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,9700,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.020619,0,[],[],1.536082,5.886598,1.536082,2026-03-12 14:33:42.163987+00:00,20260312_quotes_lifecycle3_test
4,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,260,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.384615,0.769231,0.384615,2026-03-12 14:33:42.165569+00:00,20260312_quotes_lifecycle3_test
5,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1601,PASS,[],[],count_coverage,0,0.000000,0,[],[],1.374141,2.186134,1.311680,2026-03-12 14:33:42.167282+00:00,20260312_quotes_lifecycle3_test
6,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,659,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.435508,9.711684,7.283763,2026-03-12 14:33:42.168867+00:00,20260312_quotes_lifecycle3_test
7,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,399,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.017544,2.756892,7.017544,2026-03-12 14:33:42.170364+00:00,20260312_quotes_lifecycle3_test
8,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,442,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.452489,0.452489,0.226244,2026-03-12 14:33:42.172053+00:00,20260312_quotes_lifecycle3_test
9,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,273,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.732601,1.098901,0.366300,2026-03-12 14:33:42.173597+00:00,20260312_quotes_lifecycle3_test



RETRY CURRENT rows: 0


,file,severity,issues,warns,action,processed_at_utc



BATCH MANIFEST rows: 401


,file
0,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
1,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
2,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
3,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
4,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
5,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
6,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
7,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
8,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...
9,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...


## Handoff a Agent03

Usa el mismo `RUN_ID` y el mismo `RUN_DIR` cuando pases a cobertura y causas.

In [6]:
print("RUN_DIR para Agent03:")
print(TEST_RUN_DIR)
print("\nArchivos clave:")
print("-", TEST_RUN_DIR / "quotes_agent_strict_events_current.csv")
print("-", TEST_RUN_DIR / "retry_queue_quotes_strict_current.csv")
print("-", TEST_RUN_DIR / "run_config_quotes_strict.json")
print("-", TEST_RUN_DIR / "batch_manifest_quotes_strict.csv")

RUN_DIR para Agent03:
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test

Archivos clave:
- C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_current.csv
- C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\retry_queue_quotes_strict_current.csv
- C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\run_config_quotes_strict.json
- C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\batch_manifest_quotes_strict.csv
